In [ ]:
!pip -q install openai requests beautifulsoup4

In [ ]:
import json, re, getpass, requests
from bs4 import BeautifulSoup
from openai import OpenAI

api_key = getpass.getpass("Enter OpenAI API key: ")
client = OpenAI(api_key=api_key)

MODEL = "gpt-5.4-mini"

memory = [{
    "role": "system",
    "content": (
        "You are a tiny research agent. "
        "Keep answers short, clear, and useful. "
        "If the user says they like short or detailed summaries, remember it."
    )
}]

prefs = {"summary_style": "short"}

def fetch_webpage(url: str, max_chars: int = 12000) -> str:
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
        tag.decompose()

    text = " ".join(p.get_text(" ", strip=True) for p in soup.find_all("p"))
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars] if text else "No useful content found."

tools = [{
    "type": "function",
    "name": "fetch_webpage",
    "description": "Fetch main visible text from a webpage URL",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {"type": "string", "description": "Full webpage URL"}
        },
        "required": ["url"],
        "additionalProperties": False
    }
}]

def update_pref(user_text: str):
    t = user_text.lower()
    if "short summaries" in t:
        prefs["summary_style"] = "short"
    elif "detailed summaries" in t:
        prefs["summary_style"] = "detailed"

def run_agent(user_text: str):
    update_pref(user_text)
    memory.append({
        "role": "user",
        "content": f"{user_text}\nUser summary preference: {prefs['summary_style']}."
    })

    response = client.responses.create(model=MODEL, input=memory, tools=tools)

    while True:
        calls = [x for x in response.output if x.type == "function_call"]

        if not calls:
            answer = response.output_text
            memory.append({"role": "assistant", "content": answer})
            return answer

        tool_outputs = []
        for call in calls:
            args = json.loads(call.arguments)
            result = fetch_webpage(**args)
            tool_outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": result
            })

        response = client.responses.create(
            model=MODEL,
            previous_response_id=response.id,
            input=tool_outputs
        )

url = input("Enter URL: ").strip()
prompt = f"Summarize this page in bullet points: {url}"
print(run_agent(prompt))

Enter OpenAI API key: ··········
Enter URL: https://en.wikipedia.org/wiki/Deep_learning
- Deep learning is a branch of machine learning that uses multilayer neural networks to learn from data.
- The “deep” part means the network has many layers, letting it build from simple to more abstract features.
- It can be trained in supervised, semi-supervised, or unsupervised ways.
- Common architectures include CNNs, RNNs, transformers, GANs, and deep belief networks.
- It has been successful in tasks like image recognition, speech, natural language processing, translation, medicine, and game playing.
- A major advantage is that it learns features automatically, reducing the need for hand-crafted feature engineering.
- Key training ideas include backpropagation, gradient descent, and layer-by-layer learning.
- The field grew from early neural network research and became much more effective with better algorithms, hardware, and large datasets.


In [ ]:
!pip install feedparser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.8 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=a4d2b5a0503b7cf8845bf7d48a0b86f840171893cac7662bbbf633b44631fcd4
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [ ]:
import feedparser
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# -------- CONFIG --------
FEEDS = {
    "BBC Tech": "https://feeds.bbci.co.uk/news/technology/rss.xml",
    "NYT Tech": "https://rss.nytimes.com/services/xml/rss/nyt/Technology.xml",
    "TechCrunch AI": "https://techcrunch.com/category/artificial-intelligence/feed/",
}

EMAIL_USER = "pailamadhuri92@gmail.com"
EMAIL_PASS = "acsm wzow frzq ckir"   # Use Gmail App Password
EMAIL_TO = "sravanipanchireddy2004@gmail.com"
# ------------------------


def fetch_news():
    articles = []
    seen = set()

    for source, url in FEEDS.items():
        feed = feedparser.parse(url)
        for entry in feed.entries[:5]:
            title = entry.title
            link = entry.link

            if title.lower() in seen:
                continue
            seen.add(title.lower())

            articles.append((source, title, link))

    return articles


def create_email_content(articles):
    html = "<h2>Daily News Update</h2><ul>"
    for source, title, link in articles:
        html += f"<li><b>{source}</b>: {title}<br><a href='{link}'>Read more</a></li><br>"
    html += "</ul>"
    return html


def send_email(content):
    msg = MIMEMultipart()
    msg["From"] = EMAIL_USER
    msg["To"] = EMAIL_TO
    msg["Subject"] = "Daily News Digest"

    msg.attach(MIMEText(content, "html"))

    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(EMAIL_USER, EMAIL_PASS)
        server.send_message(msg)


# -------- RUN --------
news = fetch_news()
email_content = create_email_content(news)
send_email(email_content)

print("✅ Email sent successfully!")


✅ Email sent successfully!
